In [3]:

import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.quantum_grover.grover_search import create_diffusion_operator

# Get input size from the user
n_qubits = int(input("Enter the number of input bits (2-8): "))

if not 2 <= n_qubits <= 8:
    raise ValueError("Please enter a number between 2 and 8.")

# Create the diffusion operator
diffusion = create_diffusion_operator(n_qubits)

print("\nDiffusion operator created successfully!")
print(f"Number of input qubits: {n_qubits}")
print(f"Number of possible inputs: {2 ** n_qubits}")
print(diffusion.draw(output="text"))



Diffusion operator created successfully!
Number of input qubits: 3
Number of possible inputs: 8
global phase: π
     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_2: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘


In [5]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Start with all qubits in |0>
test_circuit = QuantumCircuit(3)

# Create an equal superposition of 8 states
test_circuit.h(range(3))

# Apply the diffusion operator
test_circuit.compose(diffusion, inplace=True)

# Calculate the resulting quantum state
state = Statevector.from_instruction(test_circuit)

print("Probabilities after diffusion:")

for i, probability in enumerate(state.probabilities()):
    print(f"Input {i:03b}: {probability:.4f}")


Probabilities after diffusion:
Input 000: 0.1250
Input 001: 0.1250
Input 010: 0.1250
Input 011: 0.1250
Input 100: 0.1250
Input 101: 0.1250
Input 110: 0.1250
Input 111: 0.1250


In [9]:
import importlib
import src.quantum_grover.grover_search as grover_module

importlib.reload(grover_module)

print(hasattr(grover_module, "create_grover_circuit"))

True


In [10]:
from src.hash_functions.toy_hash import create_toy_hash_system
from src.quantum_grover.grover_search import create_grover_circuit

In [15]:

import sys
from pathlib import Path

# ==========================================
# STEP 1: SET UP PROJECT IMPORTS
# ==========================================

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.hash_functions.toy_hash import create_toy_hash_system
from src.quantum_grover.grover_search import create_grover_circuit


# ==========================================
# STEP 2: GET USER INPUT
# ==========================================

try:
    n_bits = int(input("Enter input bit size (2-8): "))
    m_bits = int(input("Enter hash output size (1-8 bits): "))

    if not 2 <= n_bits <= 8:
        raise ValueError("Input bit size must be between 2 and 8.")

    if not 1 <= m_bits <= 8:
        raise ValueError("Hash output size must be between 1 and 8.")

    # ==========================================
    # STEP 3: GENERATE THE HASH MAP
    # ==========================================

    hash_map = create_toy_hash_system(n_bits, m_bits)

    available_hashes = sorted(set(hash_map.values()))

    print("\nAvailable target hashes (decimal):", available_hashes)

    # ==========================================
    # STEP 4: GET AND VALIDATE TARGET HASH
    # ==========================================

    target_hash = int(input("Enter target hash (decimal): "))

    if target_hash not in hash_map.values():
        raise ValueError(
            f"Target hash {target_hash} does not occur in this hash map. "
            f"Choose one of: {available_hashes}"
        )

    # ==========================================
    # STEP 5: FIND CLASSICAL MATCHES
    # ==========================================

    matching_inputs = [
        value
        for value, hash_value in hash_map.items()
        if hash_value == target_hash
    ]

    print("\n========== CLASSICAL RESULTS ==========")
    print("Target hash:", target_hash)
    print("Matching inputs (decimal):", matching_inputs)
    print(
        "Matching inputs (binary):",
        [format(value, f"0{n_bits}b") for value in matching_inputs]
    )

    # ==========================================
    # STEP 6: CREATE GROVER CIRCUIT
    # ==========================================

    grover = create_grover_circuit(
        hash_map=hash_map,
        target_hash=target_hash,
        n_bits=n_bits,
    )

    print("\nGrover circuit created successfully!")
    print(grover.draw(output="text"))

except ValueError as error:
    print("\nInput or search error:", error)



--- Toy Hash System Configuration ---
Input Bit Size (n): 5 (Total inputs: 32)
Hash Output Size (m): 4 bits
-------------------------------------------------------
Input (Binary)   | Input (Dec)  | Hash Output (Binary)
-------------------------------------------------------
|00000>           | 0            | |0000>
|00001>           | 1            | |0001>
|00010>           | 2            | |0010>
|00011>           | 3            | |0011>
|00100>           | 4            | |0100>
|00101>           | 5            | |0101>
|00110>           | 6            | |0110>
|00111>           | 7            | |0111>
|01000>           | 8            | |1000>
|01001>           | 9            | |1001>
|01010>           | 10           | |1010>
|01011>           | 11           | |1011>
|01100>           | 12           | |1100>
|01101>           | 13           | |1101>
|01110>           | 14           | |1110>
|01111>           | 15           | |1111>
|10000>           | 16           | |0001>
|10001>   